# HAR Datasets EDA
Concise exploration of UCI HAR, WISDM, MotionSense, HHAR — focused on mergeability.

In [ ]:
# !pip install pandas numpy matplotlib requests

In [ ]:
import re, io, zipfile, tarfile, urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def bar(ax, counts, title):
    counts.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("samples")

---
## 1. UCI HAR
30 subjects, 6 activities, 50 Hz, **pre-segmented** 128-sample windows (561 hand-crafted features + raw inertial signals).  
Source: https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones

In [ ]:
uci_zip = DATA_DIR / "uci_har.zip"
if not uci_zip.exists():
    url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
    print("Downloading UCI HAR (~60 MB)...")
    urllib.request.urlretrieve(url, uci_zip)
    print("Done.")

uci_dir = DATA_DIR / "UCI_HAR"
if not uci_dir.exists():
    with zipfile.ZipFile(uci_zip) as zf:
        zf.extractall(uci_dir)

inner = uci_dir / "UCI HAR Dataset"
if not inner.exists():
    inner_zips = list(uci_dir.rglob("*.zip"))
    if inner_zips:
        with zipfile.ZipFile(inner_zips[0]) as zf:
            zf.extractall(uci_dir)
    inner = next(uci_dir.rglob("activity_labels.txt")).parent

print("UCI HAR root:", inner)

In [ ]:
act_labels = pd.read_csv(inner / "activity_labels.txt", sep=" ", header=None, names=["id", "activity"])
feat_raw   = pd.read_csv(inner / "features.txt", sep=" ", header=None, names=["id", "feat"])["feat"].tolist()

seen = Counter()
feats = []
for f in feat_raw:
    seen[f] += 1
    feats.append(f if seen[f] == 1 else f + f"_{seen[f]}")

def load_split(split):
    X = pd.read_csv(inner / split / f"X_{split}.txt", sep=r"\s+", header=None, names=feats)
    y = pd.read_csv(inner / split / f"y_{split}.txt", header=None, names=["label"])
    s = pd.read_csv(inner / split / f"subject_{split}.txt", header=None, names=["subject"])
    return pd.concat([X, y, s], axis=1)

uci = pd.concat([load_split("train"), load_split("test")], ignore_index=True)
uci = uci.merge(act_labels, left_on="label", right_on="id").drop(columns=["id", "label"])

print(f"Shape:    {uci.shape}")
print(f"Subjects: {uci['subject'].nunique()}")
print(uci['activity'].value_counts().to_string())

fig, ax = plt.subplots(figsize=(5, 3))
bar(ax, uci['activity'].value_counts(), "UCI HAR — activity distribution")
plt.tight_layout(); plt.show()


In [ ]:
raw_signals = list((inner / "train" / "Inertial Signals").glob("*.txt"))
print("Raw signal files:", [p.name for p in raw_signals])
sample_raw = pd.read_csv(raw_signals[0], sep=r"\s+", header=None)
print(f"  Each file shape: {sample_raw.shape}  (n_windows × 128 samples)")


---
## 2. WISDM v1.1
36 subjects, 6 activities, ~20 Hz, **raw** accelerometer only (x, y, z).  
Source: https://www.cis.fordham.edu/wisdm/dataset.php

In [ ]:
wisdm_dir = DATA_DIR / "WISDM"
wisdm_dir.mkdir(exist_ok=True)
wisdm_path = wisdm_dir / "WISDM_ar_v1.1_raw.txt"

if not wisdm_path.exists():
    # Fordham original is 404; use GitHub mirror of the same file
    url = "https://raw.githubusercontent.com/brangerbriz/machine-learning-docs/master/data/WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt"
    print("Downloading WISDM v1.1 raw data...")
    urllib.request.urlretrieve(url, wisdm_path)
    print("Done.")

print("WISDM raw file:", wisdm_path)

In [ ]:

# Robustly parse the non-standard CSV (each line ends with ';', some lines malformed)
def load_wisdm(path):
    rows = []
    with open(path, errors="ignore") as f:
        for line in f:
            line = re.sub(r"[;\s]+$", "", line.strip())
            parts = line.split(",")
            if len(parts) != 6:
                continue
            try:
                rows.append({
                    "subject":   int(parts[0]),
                    "activity":  parts[1].strip(),
                    "timestamp": int(parts[2]),
                    "x": float(parts[3]),
                    "y": float(parts[4]),
                    "z": float(parts[5]),
                })
            except ValueError:
                pass
    return pd.DataFrame(rows)

wisdm = load_wisdm(wisdm_path)

print(f"Shape:    {wisdm.shape}")
print(f"Subjects: {wisdm['subject'].nunique()}")
print(wisdm['activity'].value_counts().to_string())

# Estimate sampling rate from timestamps (nanoseconds)
user1 = wisdm[wisdm['subject'] == 1].sort_values('timestamp')
diffs = user1['timestamp'].diff().dropna()
median_diff_ns = diffs[diffs > 0].median()
print(f"\nEstimated sampling rate: ~{1e9 / median_diff_ns:.0f} Hz")

fig, ax = plt.subplots(figsize=(5, 3))
bar(ax, wisdm['activity'].value_counts(), "WISDM — activity distribution")
plt.tight_layout(); plt.show()

---
## 3. MotionSense
24 subjects, 6 activities, 50 Hz, **raw** device motion: accelerometer + gyroscope + attitude.  
Source: https://github.com/mmalekzadeh/motion-sense

In [ ]:
ms_dir = DATA_DIR / "MotionSense"
ms_dir.mkdir(exist_ok=True)
ms_zip  = ms_dir / "A_DeviceMotion_data.zip"
ms_root = ms_dir / "A_DeviceMotion_data"

if not ms_zip.exists():
    url = "https://github.com/mmalekzadeh/motion-sense/raw/master/data/A_DeviceMotion_data.zip"
    print("Downloading MotionSense (~25 MB)...")
    urllib.request.urlretrieve(url, ms_zip)
    print("Done.")


if not ms_root.exists():
    with zipfile.ZipFile(ms_zip) as zf:
        zf.extractall(ms_root.parent)

# Walk activity_trial/sub_N.csv layout
def load_motionsense(root):
    dfs = []
    for act_dir in sorted(root.iterdir()):
        if not act_dir.is_dir():
            continue
        act = act_dir.name.rsplit("_", 1)[0]   # e.g. "dws_1" -> "dws"
        for csv_file in sorted(act_dir.glob("sub_*.csv")):
            subject = int(re.search(r"\d+", csv_file.stem).group())
            df = pd.read_csv(csv_file)
            df["activity"] = act
            df["subject"]  = subject
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

ms = load_motionsense(ms_root)

ACT_MAP = {"dws": "downstairs", "ups": "upstairs", "sit": "sit",
           "std": "stand", "wlk": "walk", "jog": "jog"}
ms["activity"] = ms["activity"].map(ACT_MAP)

print(f"Shape:    {ms.shape}")
print(f"Subjects: {ms['subject'].nunique()}")
print(f"Columns:  {[c for c in ms.columns if c not in ('activity','subject')]}")
print(ms['activity'].value_counts().to_string())

fig, ax = plt.subplots(figsize=(5, 3))
bar(ax, ms['activity'].value_counts(), "MotionSense — activity distribution")
plt.tight_layout(); plt.show()

---
## 4. HHAR (Heterogeneity Activity Recognition)
9 users, 6 activities, **variable** sampling rate per device (100–200 Hz), raw acc + gyro from multiple phone/watch models.  
Source: https://archive.ics.uci.edu/dataset/344/heterogeneity+activity+recognition

In [ ]:
import subprocess

hhar_dir = DATA_DIR / "HHAR"
hhar_dir.mkdir(exist_ok=True)
phones_acc = hhar_dir / "Phones_accelerometer.csv"

if not phones_acc.exists():
    wrapped = hhar_dir / "Phones_accelerometer.csv.zip"
    if wrapped.exists():
        print("Extracting existing zip...")
        with zipfile.ZipFile(wrapped) as zf:
            zf.extractall(hhar_dir)
    else:
        print("Downloading Phones_accelerometer.csv via Kaggle CLI...")
        subprocess.run([
            ".venv/bin/kaggle", "datasets", "download",
            "-d", "chumajin/heterogeneity-human-activity-recognition-dataset",
            "-f", "Phones_accelerometer.csv",
            "-p", str(hhar_dir),
        ], check=True)
        with zipfile.ZipFile(hhar_dir / "Phones_accelerometer.csv.zip") as zf:
            zf.extractall(hhar_dir)

print("Using:", phones_acc)

In [ ]:
hhar = pd.read_csv(phones_acc)
# Columns: Index, Arrival_Time, Creation_Time, x, y, z, User, Model, Device, gt
hhar = hhar.rename(columns={"gt": "activity", "User": "subject"})
hhar = hhar[hhar['activity'].notna() & (hhar['activity'] != 'null')]

print(f"Shape:    {hhar.shape}")
print(f"Subjects: {hhar['subject'].nunique()}")
print(f"Devices:  {hhar['Device'].nunique()} unique  — {hhar['Model'].unique().tolist()}")
print(hhar['activity'].value_counts().to_string())

# Sampling rate varies by device; estimate from a single device
dev0 = hhar[hhar['Device'] == hhar['Device'].iloc[0]].sort_values('Arrival_Time')
sr = 1e9 / dev0['Arrival_Time'].diff().median()
print(f"\nEstimated SR for device 0: ~{sr:.0f} Hz")

fig, ax = plt.subplots(figsize=(5, 3))
bar(ax, hhar['activity'].value_counts(), "HHAR — activity distribution (phones acc)")
plt.tight_layout(); plt.show()

---
## 5. Cross-dataset comparison

In [ ]:
summary = pd.DataFrame([
    {"Dataset":     "UCI HAR",
     "Subjects":    uci['subject'].nunique(),
     "Samples":     len(uci),
     "Activities":  uci['activity'].nunique(),
     "Sampling Hz": "50 (pre-windowed)",
     "Sensors":     "acc + gyro (engineered + raw)",
     "Format":      "128-sample windows"},
    {"Dataset":     "WISDM",
     "Subjects":    wisdm['subject'].nunique(),
     "Samples":     len(wisdm),
     "Activities":  wisdm['activity'].nunique(),
     "Sampling Hz": "~20",
     "Sensors":     "acc only",
     "Format":      "raw"},
    {"Dataset":     "MotionSense",
     "Subjects":    ms['subject'].nunique(),
     "Samples":     len(ms),
     "Activities":  ms['activity'].nunique(),
     "Sampling Hz": "50",
     "Sensors":     "acc + gyro + attitude (roll/pitch/yaw)",
     "Format":      "raw"},
    {"Dataset":     "HHAR",
     "Subjects":    hhar['subject'].nunique(),
     "Samples":     len(hhar),
     "Activities":  hhar['activity'].nunique(),
     "Sampling Hz": "100–200 (device-dependent)",
     "Sensors":     "acc + gyro (separate files)",
     "Format":      "raw"},
]).set_index("Dataset")

print(summary.to_string())

In [ ]:
# Activity label mapping across datasets
label_map = pd.DataFrame({
    "Canonical":    ["walk",      "upstairs",         "downstairs",         "sit",     "stand",    "jog",    "bike",  "lay"],
    "UCI HAR":      ["WALKING",   "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "—",      "—",     "LAYING"],
    "WISDM":        ["Walking",   "Upstairs",         "Downstairs",         "Sitting", "Standing", "Jogging","—",     "—"],
    "MotionSense":  ["walk",      "upstairs",         "downstairs",         "sit",     "stand",    "jog",    "—",     "—"],
    "HHAR":         ["walk",      "stairsup",         "stairsdown",         "sit",     "stand",    "—",      "bike",  "—"],
})
print(label_map.to_string(index=False))

In [ ]:
COMMON = ["walk", "upstairs", "downstairs", "sit", "stand"]

canon = {
    "UCI HAR":     {"WALKING":"walk","WALKING_UPSTAIRS":"upstairs","WALKING_DOWNSTAIRS":"downstairs","SITTING":"sit","STANDING":"stand"},
    "WISDM":       {"Walking":"walk","Upstairs":"upstairs","Downstairs":"downstairs","Sitting":"sit","Standing":"stand"},
    "MotionSense": {"walk":"walk","upstairs":"upstairs","downstairs":"downstairs","sit":"sit","stand":"stand"},
    "HHAR":        {"walk":"walk","stairsup":"upstairs","stairsdown":"downstairs","sit":"sit","stand":"stand"},
}

frames = {
    "UCI HAR":     uci,
    "WISDM":       wisdm,
    "MotionSense": ms,
    "HHAR":        hhar,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, df) in zip(axes, frames.items()):
    mapped = df['activity'].map(canon[name]).dropna()
    counts = mapped.value_counts().reindex(COMMON, fill_value=0)
    pct = counts / counts.sum() * 100
    pct.plot.bar(ax=ax, color="steelblue", rot=30)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("% of samples" if ax == axes[0] else "")
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

fig.suptitle("Common 5-class distribution (% of dataset)", y=1.02)
plt.tight_layout(); plt.show()


## Merge notes

**Sensor overlap**  
- All four datasets have **accelerometer (x, y, z)** — the only common modality.  
- WISDM has *only* accelerometer; if you want gyro or attitude, WISDM drops out.

**Activity overlap**  
- 5 clean common classes: `walk, upstairs, downstairs, sit, stand`.  
- `jog` exists in WISDM + MotionSense only; `lay` in UCI HAR only; `bike` in HHAR only.  
- Drop or keep as an extra class depending on study focus.

**Sampling rate**  
- UCI HAR & MotionSense: 50 Hz.  
- WISDM: ~20 Hz — needs upsampling or you resegment all datasets at 20 Hz.  
- HHAR: 100–200 Hz device-dependent — needs downsampling + interpolation.

**Segmentation**  
- UCI HAR ships as 128-sample windows; for merging you'd use the **raw inertial signals** (`Inertial Signals/`) and re-window uniformly.  
- All others are raw — apply the same sliding window (e.g. 128 samples at 50 Hz = 2.56 s, 50% overlap) after resampling.

**Minimum viable merge path**  
1. Resample everything to **50 Hz** (decimate HHAR, upsample WISDM).  
2. Use **accelerometer only** (x, y, z) from the raw/inertial signals.  
3. Apply a uniform sliding window (e.g. 128 × 50% overlap).  
4. Map to the **5 common labels**; discard `jog`, `bike`, `lay` or add as optional classes.  
5. Add a `source` column for domain adaptation experiments.